# Foundational models with LangChain

Initialise chat models (local Ollama and Google Gemini), call one, then wrap it in an agent and stream the answer.

- **Run:** double-click `run_1_1.cmd` in this folder → *Run → Run All Cells* (from the top — later cells reuse `model`, `agent`, `show`).
- **Needs:** only `GOOGLE_API_KEY` (from `.env`, Colab Secrets or pasted below). No OpenAI or Anthropic key is used.

**Idea:** the model is the agent's brain — every later lesson hangs off it.

**What I learned:**
- An answer is an object, not a string: read `response_metadata` and `usage_metadata` — tokens are what you pay for.
- Four dials change behaviour (`temperature`, `max_tokens`, `timeout`, `max_retries`) — change one at a time.
- An agent = a model + instructions, called with a *list* of messages — that list is what memory builds on later.
- Control it, simplest first: system prompt → examples (few-shot) → `response_format` schema. Start small and cheap.


## Setup

In [ ]:
!pip install -q langchain langchain-ollama langchain-google-genai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # local run: reads the .env file (returns False if none is found)

# Colab run: read the key from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..._here") count as missing
    return bool(value) and not value.startswith("your_")

# Still missing? Paste it here (hidden input).
if not is_real(os.getenv("GOOGLE_API_KEY")):
    try:
        value = getpass("GOOGLE_API_KEY is missing - paste it (Enter to skip): ").strip()
    except Exception:
        value = ""
    if value:
        os.environ["GOOGLE_API_KEY"] = value

print("GOOGLE_API_KEY:", "set" if is_real(os.getenv("GOOGLE_API_KEY")) else "MISSING")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY is missing - paste it (Enter to skip): ··········
GOOGLE_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


## Initialising and invoking a model
First a **local** model served by Ollama (no API key needed). If Ollama cannot start, the notebook falls back to Gemini.

> **`model.invoke(prompt)`**: sends a prompt to the model and returns its answer as an `AIMessage`.

In [ ]:
import json, os, shutil, subprocess, sys, time, urllib.request

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "llama3.2:1b"  # locally you can also use "llama3.1:8b" or "gpt-oss:20b"

def server_up():
    try:
        urllib.request.urlopen(OLLAMA_URL, timeout=2)
        return True
    except OSError:
        return False

def ensure_ollama():
    """Make sure the Ollama server runs and MODEL is pulled. Returns True on success."""
    try:
        # 1) Start the server only if it is not already running
        if not server_up():
            if shutil.which("ollama") is None:
                if sys.platform.startswith("linux"):  # Colab / Linux: install Ollama
                    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
                else:  # Windows / macOS: install the app manually
                    raise RuntimeError("Ollama is not installed: https://ollama.com/download")
            log = open("ollama_server.log", "w")  # keep the server output to explain any failure
            proc = subprocess.Popen(
                ["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT,
                env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"}, start_new_session=True,
            )
            for _ in range(90):  # first start can be slow
                if server_up() or proc.poll() is not None:
                    break
                time.sleep(1)
            if not server_up():
                log.flush()
                print("---- ollama_server.log ----")
                print(open("ollama_server.log").read()[-3000:])
                raise RuntimeError("Ollama server did not start (see the log above)")
        print("Ollama server is up")

        # 2) Download the model only if it is not already pulled
        with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags") as r:
            installed = [m["name"] for m in json.load(r)["models"]]
        if MODEL not in installed:
            subprocess.run(["ollama", "pull", MODEL], check=True)
        print(f"Model ready: {MODEL}")
        return True
    except Exception as e:
        print(f"Local Ollama unavailable: {e}")
        return False

OLLAMA_OK = ensure_ollama()

Local Ollama unavailable: Command 'curl -fsSL https://ollama.com/install.sh | sh' returned non-zero exit status 1.


In [ ]:
from langchain_ollama import ChatOllama
from langchain.chat_models import init_chat_model

if OLLAMA_OK:
    model = ChatOllama(model=MODEL, base_url=OLLAMA_URL, temperature=0)
else:
    # Fallback so the rest of the notebook still runs (needs GOOGLE_API_KEY)
    print("Ollama unavailable -> using Gemini for this section instead")
    model = init_chat_model("google_genai:gemini-3.1-flash-lite", temperature=0)

Ollama unavailable -> using Gemini for this section instead


In [ ]:
response = model.invoke("What's the capital of the Moon?")

response

AIMessage(content=[{'type': 'text', 'text': 'The Moon does not have a capital because it is not a country and has no permanent human government or population. \n\nUnder the **Outer Space Treaty of 1967**, which has been signed by all major spacefaring nations, no nation can claim sovereignty over the Moon or any other celestial body. Therefore, it is considered the "province of all mankind" rather than a territory with political borders or administrative centers.', 'extras': {'signature': 'EnEKbwFpFH0T12I+ROfWSK6rEjhS3GE2EweRJZK4ALrDpVlQq79o4vdrtbhGq3sNBqL0FYsqwnqeHe7p97U6cJemmenL7UxArabUmS4qeVMtUbKxhNXYl81vOHZ6Z/uEKF1u4ffh2ZAoDTpynQ0uLTtkEg=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0cfcd-b13c-7843-98d4-3ba28ac141e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 86, 'total_tokens': 96, 'input_token_

In [ ]:
print(response.text)

The Moon does not have a capital because it is not a country and has no permanent human government or population. 

Under the **Outer Space Treaty of 1967**, which has been signed by all major spacefaring nations, no nation can claim sovereignty over the Moon or any other celestial body. Therefore, it is considered the "province of all mankind" rather than a territory with political borders or administrative centers.


In [ ]:
from pprint import pprint

pprint(response.response_metadata)

{'finish_reason': 'STOP',
 'model_name': 'gemini-3.1-flash-lite',
 'model_provider': 'google_genai',
 'safety_ratings': []}


> **`response.usage_metadata`**: number of tokens used by the call (input, output, total). Tokens are what you pay for, so get into the habit of reading them.

In [ ]:
print(response.usage_metadata)
print("Model that answered:", response.response_metadata.get("model_name"))

{'input_tokens': 10, 'output_tokens': 86, 'total_tokens': 96, 'input_token_details': {'cache_read': 0}}
Model that answered: gemini-3.1-flash-lite


## Customising your Model
`init_chat_model` builds a model from a `provider:model` string and passes extra kwargs (here `temperature`) to it.

> **`init_chat_model("provider:model", **kwargs)`**: creates a chat model from a name string. **`temperature`** controls randomness: 0 = predictable, 1 = more creative.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "google_genai:gemini-3.1-flash-lite",
    # Kwargs passed to the model:
    temperature=1.0,
)

response = model.invoke("What's the capital of the Moon?")
print(response.text)

The Moon does not have a capital because it is not a country and has no permanent human government. Under the **Outer Space Treaty of 1967**, which has been signed by all major spacefaring nations, no nation is permitted to claim sovereignty over the Moon or any other celestial body.

While there are various research bases planned for the future—such as the Artemis program's proposed **Artemis Base Camp**—these are international or national outposts rather than capitals.


### The four dials
| Setting | What it does |
|---|---|
| `temperature` | Randomness: higher = more creative, lower = more predictable. |
| `max_tokens` | Caps how long the response can be. |
| `timeout` | Seconds to wait before giving up on a request. |
| `max_retries` | How many times to try again when a request fails. |

`timeout` and `max_retries` matter: you call someone else's service over the internet, and it will sometimes be slow or fail.

In [ ]:
model = init_chat_model(
    "google_genai:gemini-3.1-flash-lite",
    temperature=1.0,  # randomness
    max_tokens=1000,  # cap on the answer length
    timeout=30,       # seconds before giving up
    max_retries=2,    # retries if the request fails
)

response = model.invoke("What's the capital of the Moon?")
print(response.text)

The Moon does not have a capital city, a government, or any permanent inhabitants. Under the **Outer Space Treaty of 1967**, no nation can claim sovereignty over the Moon or any other celestial body, meaning it has no political borders or administrative centers.


In [ ]:
# Change one thing at a time: same question, low vs high temperature.
# Run this cell several times: the low temperature answer barely changes, the high one does.
question = "Answer in one sentence: what is the capital of the Moon?"

for temp in (0.0, 1.5):
    m = init_chat_model("google_genai:gemini-3.1-flash-lite", temperature=temp)
    print(f"\ntemperature = {temp}\n{m.invoke(question).text}")


temperature = 0.0
The Moon has no capital because it is a celestial body with no government, sovereign nations, or permanent human settlements.

temperature = 1.5
The Moon has no capital as it is an uninhabited celestial body with no sovereign government or permanent human settlement.


## Model Providers
Any provider from https://docs.langchain.com/oss/python/integrations/chat can be used the same way. Here the same Gemini model is created with its provider class instead:

> **Model agnostic:** moving an agent from one provider to another is close to swapping the model name, with the API key kept in `.env` and never in the code. Do not marry a provider: build so you can leave. Start with a small, cheap model and move to a bigger one only when the cheap one has actually failed.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

response = model.invoke("What's the capital of the Moon?")
print(response.text)

The Moon does not have a capital because it is not a country and has no permanent human government or population. 

According to the **1967 Outer Space Treaty**, which has been signed by all major spacefaring nations, no nation can claim sovereignty over the Moon or any celestial body. Therefore, there are no cities, states, or administrative centers on the Moon.


## Initialising and invoking an agent
Two equivalent ways to create the agent (both use Gemini). Run **one** of them.

> **`create_agent(model, system_prompt=...)`**: wraps a model in an agent. You call it with a list of messages: **`HumanMessage`** is what the user says, **`AIMessage`** is what the model said before.

In [ ]:
from langchain.agents import create_agent

# Option 1: from a model object (the Gemini model defined above)
agent = create_agent(model=model)

In [ ]:
# Option 2: from a model string
agent = create_agent(model="google_genai:gemini-3.1-flash-lite")

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?")]}
)

In [ ]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='6afb93d5-d5aa-489f-9f91-45faa1d285fc'),
              AIMessage(content=[{'type': 'text', 'text': 'The Moon does not have a capital because it has no government, no countries, and no permanent human residents. Under the **Outer Space Treaty of 1967**, which has been signed by all major spacefaring nations, no nation can claim sovereignty over the Moon or any other celestial body.\n\nWhile there are many proposed sites for future human bases (such as the Shackleton Crater at the lunar south pole, which is a prime target for NASA’s Artemis missions), none of these carry the status of a capital city.', 'extras': {'signature': 'EnEKbwFpFH0TnUJlVjS4CvPh0N0ql2SZpHXBevDKZRAVPJbv0C70JRtHxgwOo9NHq+qHw2tWmbuN2vZ4/tBfN+UBFA7scmraTZGW1OZNQqCxhyRDxaXWimzONwbdulwNB7wmkbwA1vlPfcyvhI+se/DCAg=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-

In [ ]:
print(response['messages'][-1].text)

The Moon does not have a capital because it has no government, no countries, and no permanent human residents. Under the **Outer Space Treaty of 1967**, which has been signed by all major spacefaring nations, no nation can claim sovereignty over the Moon or any other celestial body.

While there are many proposed sites for future human bases (such as the Shackleton Crater at the lunar south pole, which is a prime target for NASA’s Artemis missions), none of these carry the status of a capital city.


In [ ]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What's the capital of the Moon?"),
        AIMessage(content="The capital of the Moon is Luna City."),
        HumanMessage(content="Interesting, tell me more about Luna City"),
    ]}
)

pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='f14892c7-20d6-40b4-9425-551926c8f7a9'),
              AIMessage(content='The capital of the Moon is Luna City.', additional_kwargs={}, response_metadata={}, id='8b318311-b903-4324-9f62-dc5b60132c7f', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interesting, tell me more about Luna City', additional_kwargs={}, response_metadata={}, id='eb49dab2-24fe-4c99-b89a-81bc6b313b6e'),
              AIMessage(content=[{'type': 'text', 'text': 'It is important to clarify that **Luna City is a fictional location.** While the Moon is a real place, it has no government, no borders, and no cities—and therefore, no capital city.\n\nHowever, the concept of "Luna City" is a staple of science fiction and space exploration literature. Here is why the name is so iconic and how it has been imagined:\n\n### 1. The Fictional "Luna City"\nThe most famous depiction co

## System prompt
A **system prompt** sets the model's role and rules before the conversation starts. The same question gets a different answer with and without it.

> **`SystemMessage`** / **`system_prompt`**: instructions given to the model before the conversation (its role, tone and rules).

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

QUESTION = "What's the capital of the Moon?"

SYSTEM_PROMPT = (
    "You are a strict science tutor. "
    "The Moon has no capital: correct any false premise politely. "
    "Answer in exactly two sentences and start with 'Short answer:'."
)

def show(title, text):
    print(f"\n{'=' * 12} {title} {'=' * 12}\n{text}")

# Model level: pass the system prompt as the first message
without_system = model.invoke([HumanMessage(QUESTION)])
with_system = model.invoke([SystemMessage(SYSTEM_PROMPT), HumanMessage(QUESTION)])

show("WITHOUT system prompt", without_system.text)
show("WITH system prompt", with_system.text)


============ WITHOUT system prompt ============
The Moon does not have a capital because it has no government, no countries, and no permanent human population. 

Under the **Outer Space Treaty of 1967**, which has been signed by all major spacefaring nations, no country is allowed to claim sovereignty over the Moon or any other celestial body. Therefore, it remains international territory and does not have political borders or capital cities.

============ WITH system prompt ============
Short answer: The Moon does not have a capital because it is a celestial body without human governance or nations. Please note that no lunar territory is divided into states or cities in a political sense.


In [ ]:
# Agent level: pass it with the system_prompt argument
agent_plain = create_agent(model=model)
agent_tutor = create_agent(model=model, system_prompt=SYSTEM_PROMPT)

msg = {"messages": [HumanMessage(QUESTION)]}

show("AGENT without system prompt", agent_plain.invoke(msg)["messages"][-1].text)
show("AGENT with system prompt", agent_tutor.invoke(msg)["messages"][-1].text)


============ AGENT without system prompt ============
The Moon does not have a capital because it has no government, no countries, and no permanent human population. Under international law, specifically the **Outer Space Treaty of 1967**, no nation can claim sovereignty over the Moon or any other celestial body.

While there are several research bases and scientific outposts planned for the future—such as NASA's Artemis program base camp or international lunar research stations—these are scientific facilities rather than political capitals.

============ AGENT with system prompt ============
Short answer: The Moon does not have a capital because it is a celestial body with no human government or sovereign territory. It is essential to remember that astronomical objects do not function like terrestrial nations and therefore lack such administrative centers.


In [ ]:
# Change only the system prompt: same question, different behaviour
PIRATE_PROMPT = "You are a pirate. Answer in one sentence, in pirate speak."

agent_pirate = create_agent(model=model, system_prompt=PIRATE_PROMPT)

show("AGENT with pirate prompt", agent_pirate.invoke(msg)["messages"][-1].text)


============ AGENT with pirate prompt ============
The moon be naught but a barren rock floatin' in the black, so it don't hold no capital for a scurvy dog like ye to visit!


### Show it examples (few-shot)
A couple of input and output pairs inside the system prompt teach the model the exact format you want. A rambling answer becomes a one-word answer.

In [ ]:
FEWSHOT_PROMPT = """
Answer with the city name only.

Q: What is the capital of France?
A: Paris

Q: What is the capital of Japan?
A: Tokyo
"""

agent_fewshot = create_agent(model=model, system_prompt=FEWSHOT_PROMPT)

q = {"messages": [HumanMessage("What is the capital of Spain?")]}

show("WITHOUT examples", agent_plain.invoke(q)["messages"][-1].text)
show("WITH two examples", agent_fewshot.invoke(q)["messages"][-1].text)


============ WITHOUT examples ============
The capital of Spain is **Madrid**.

============ WITH two examples ============
Madrid


## Streaming Output

> **`agent.stream(..., stream_mode="messages")`**: returns the answer piece by piece (token by token) instead of waiting for the full reply.

Agents answer in seconds, sometimes minutes. Streaming does not make the answer faster, it reduces the **felt wait**, because the first words appear immediately.

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages",
):
    # token is a message chunk with token content
    # metadata contains which node produced the token
    if token.text:  # only print chunks that carry text
        print(token.text, end="", flush=True)

In science fiction and speculative futurism, **Luna City** is the quintessential "capital of the Moon." While there is no actual city on the Moon today, the concept has been a staple of literature, gaming, and space advocacy for nearly a century.

Depending on which "universe" you are looking at, Luna City takes on a different form. Here is the breakdown of the most iconic versions of Luna City and the reality of lunar settlement.

---

### 1. The Literary Classic: Robert A. Heinlein’s *The Moon Is a Harsh Mistress*
The most famous version of Luna City appears in Robert A. Heinlein’s 1966 masterpiece. In this universe:
*   **Location:** It is a massive underground complex, primarily located in the crater **Tycho**.
*   **Infrastructure:** Because of the harsh lunar surface (radiation, micro-meteorites, and temperature extremes), the city is built deep underground. It relies on "honeycomb" tunnels carved into the rock.
*   **Atmosphere:** It is a libertarian, frontier-style society. It 

In [ ]:
import time

# Streaming does not reduce the total wait, it reduces the FELT wait
start = time.time()
first_token = None

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages",
):
    if token.text:
        if first_token is None:
            first_token = time.time() - start
        print(token.text, end="", flush=True)

print(f"\n\nFirst words after {first_token:.1f}s, full answer after {time.time() - start:.1f}s")

"Luna City" is a fascinating concept because it bridges the gap between **hard science fiction, speculative urban planning, and actual lunar exploration goals.**

Depending on whether you are looking for the fictional setting of classic literature or the future plans of modern space agencies, "Luna City" takes on a few different meanings.

---

### 1. The Fictional "Luna City" (Robert A. Heinlein)
The most famous incarnation of Luna City comes from Robert A. Heinlein’s 1951 classic novel, ***The Moon is a Harsh Mistress***. In this vision:
*   **The Setting:** Luna City is a sprawling subterranean complex built into the walls of an extinct volcanic tube.
*   **The Lifestyle:** It is a rough-and-tumble "frontier" town. Residents are called "Loonies." It is a society defined by a libertarian culture, where oxygen and water are the primary currencies, and the city operates under a decentralized, high-tech, yet gritty government.
*   **Cultural Significance:** This version of Luna City set

## Structured output
Instead of free text, an agent can return data in a **fixed structure** that your code can read directly.

First approach: describe the structure in the system prompt. The answer follows the layout, but it is still plain text that you would have to parse.

In [ ]:
SCIFI_PROMPT = """
You are a science fiction writer, create a space capital city at the user's request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries
"""

agent_text = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    system_prompt=SCIFI_PROMPT,
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent_text.invoke({"messages": [question]})
print(response["messages"][-1].text)

Name: Artemis Prime

Location: Situated within the permanently shadowed rim of the Shackleton Crater, utilizing the lunar south pole’s "peaks of eternal light" for solar energy harvesting.

Vibe: Brutalist, ethereal, clinical.

Economy: Helium-3 extraction, deep-space telecommunications relay, and zero-G medical research.


Better approach: give the agent a **schema** with `response_format`. The result is a Python object with typed fields, stored in `response["structured_response"]`.

> **`BaseModel`** (from `pydantic`): a class whose attributes define the fields you want (name, type). **`response_format=Schema`**: tells the agent to return an object of that class.

In [ ]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    system_prompt="You are a science fiction writer, create a capital city at the user's request.",
    response_format=CapitalInfo,
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke({"messages": [question]})

response["structured_response"]

CapitalInfo(name='Selene Prime', location='The Shackleton Crater, lunar south pole', vibe='Brutalist tranquility encased in shimmering geodesic domes, illuminated by eternal twilight and Earth-shine.', economy='Extraction of Helium-3 and high-end orbital manufacturing fueled by low-gravity precision engineering.')

In [ ]:
response["structured_response"].name

'Selene Prime'

In [ ]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")

Selene Prime is a city located at The Shackleton Crater, lunar south pole


## Summary of functions

| Function / class | Simple definition |
|---|---|
| `load_dotenv()` | Reads the `.env` file and puts its keys in the environment. |
| `ChatOllama(model, base_url, temperature)` | Chat model that runs **locally** through Ollama. |
| `init_chat_model("provider:model", **kwargs)` | Creates a chat model from a name string, for any provider. |
| `ChatGoogleGenerativeAI(model=...)` | Gemini chat model created with its provider class. |
| `model.invoke(input)` | Sends a text or a list of messages and returns **one** `AIMessage`. |
| `response.text` | The answer as plain text (safer than `.content`, which can be a list of blocks). |
| `response.response_metadata` | Details about the call: model name, token usage, finish reason. |
| `HumanMessage` | A message written by the user. |
| `AIMessage` | A message written by the model (used to give earlier answers as history). |
| `SystemMessage` | Instructions that set the model's role and rules. |
| `create_agent(model, system_prompt=...)` | Builds an agent around a model, with an optional system prompt. |
| `agent.invoke({"messages": [...]})` | Runs the agent on a conversation and returns a dict with the `messages` list. |
| `agent.stream(..., stream_mode="messages")` | Gives the answer token by token as it is generated. |
| `BaseModel` (pydantic) | Class that defines the fields (name, type) of a structured answer. |
| `response_format=Schema` | Makes the agent return an object of that class instead of free text. |
| `response["structured_response"]` | The structured object; read its fields with `.name`, `.location`... |
| `temperature` | Randomness of the answer: 0 = predictable, 1 = more creative. |
| `max_tokens` | Maximum length of the response. |
| `timeout` | Seconds to wait before giving up on a request. |
| `max_retries` | Number of new attempts when a request fails. |
| `response.usage_metadata` | Tokens used by the call (input, output, total): what you pay for. |
| Few-shot examples | Input/output pairs in the system prompt that fix the answer format. |
| `ensure_ollama()`, `server_up()`, `show()` | Helper functions written in this notebook: start/check Ollama, and print a titled answer. |

By
> Kassouar Fatima                   23-09-2026:21:30

